In [1]:
import os

# isort: off
import xarray as xr
import pint_xarray
import cf_xarray.units
from pint_xarray import unit_registry as ureg
# isort: on

import pandas as pd
from numpy.typing import ArrayLike
from pint import Quantity

from utils import (
    PLASTIC_SHAPE_FACTORS,
    PLASTIC_SIZES,
    PROCESSED_DIR,
    RESULTS_DIR,
    clean_styler,
    powerlaw_compute_bin_edges,
    powerlaw_compute_c,
    powerlaw_compute_mass,
    powerlaw_compute_number,
)

# Overwrite outputs
OVERWRITE = True


def harmonize_size(
    data: pd.DataFrame,
    size_min: Quantity,
    size_max: Quantity,
    alpha: float,
    shape_factor: float | ArrayLike,
    density: Quantity,
) -> pd.DataFrame:
    """Extrapolate microplastic number and mass to new size range

    Args:
        data: DataFrame containing observed particle counts and sampled size ranges
        xmin, xmax: Bounds of new size range
        alpha: Power law parameter alpha, the exponent (must be > 1)
        c: Power law parameter C, the scaling factor
        shape_factor: Ratio of particle volume to cube of particle size i.e. volume / x^3.
            A sphere of diameter x has shape_factor = pi / 6.
        density: Particle density

    Returns:
        DataFrame with
    """

    # Exclude obs with unknown size range
    size_cols = ["size_low", "size_high"]
    size_units = "µm"
    for col in size_cols:
        if col not in data:
            raise ValueError(f"data must have column '{col}'")
    data = data.dropna(subset=size_cols).copy()

    # Identify number and mass columns
    def get_col(data: pd.DataFrame, options: list[str]) -> str:
        col = [x for x in options if x in data.columns]
        if not any(col):
            raise ValueError(f"data must have one of columns {options}")
        if len(col) > 1:
            raise ValueError(f"data must have only one of columns {col}")
        return col[0]

    number_col = get_col(data, ["particle/m3", "particle/m2/d"])
    mass_col = get_col(data, ["ug/m3", "t/km2/yr"])
    number_units = number_col
    mass_units = mass_col

    # Ensure units
    size_min = size_min.to(size_units)
    size_max = size_max.to(size_units)
    density = density.to("g/cm3")

    # Compute powerlaw parameters
    n_observed = ureg(number_units) * data[number_col].to_numpy()
    xmin = ureg(size_units) * data["size_low"].to_numpy()
    xmax = ureg(size_units) * data["size_high"].to_numpy()
    c = powerlaw_compute_c(n=n_observed, xmin=xmin, xmax=xmax, alpha=alpha)

    # Extrapolate to new size range
    number = powerlaw_compute_number(xmin=size_min, xmax=size_max, alpha=alpha, c=c)
    mass = powerlaw_compute_mass(
        xmin=size_min,
        xmax=size_max,
        alpha=alpha,
        c=c,
        density=density,
        shape_factor=shape_factor,
    )
    number = number.to(number_units)
    mass = mass.to(mass_units)

    # Compute mass per particle
    ng_per_particle = (mass / number).to("ng/particle")

    # Update data
    data[number_col] = number.magnitude
    data[mass_col] = mass.magnitude
    data["size_low"] = size_min.magnitude
    data["size_high"] = size_max.magnitude
    data["alpha"] = alpha
    data["c"] = c.magnitude
    data["c_units"] = "{:cf}".format(c.units)
    data["shape_factor"] = shape_factor
    data["density"] = density.magnitude
    data["density_units"] = "{:cf}".format(density.units)
    data["ng/particle"] = ng_per_particle.magnitude

    return data


def harmonize_or_load(
    obs_name: str,
    out_name: str,
    size_min: Quantity,
    size_max: Quantity,
    alpha: float,
    shape_factor: float,
    density: Quantity,
    out_dir: str = RESULTS_DIR,
    overwrite: bool = False,
) -> pd.DataFrame:
    """Wrapper to skip processing if output file already exists"""

    out_path = os.path.join(out_dir, out_name)
    if os.path.exists(out_path) and not overwrite:
        print(f"File exists: {out_path}")
        return pd.read_csv(out_path)

    # Load processed observations
    obs_path = os.path.join(PROCESSED_DIR, obs_name)
    obs = pd.read_csv(obs_path)

    # Extrapolate to new size range
    harmonized = harmonize_size(
        obs,
        size_min=size_min,
        size_max=size_max,
        alpha=alpha,
        shape_factor=shape_factor,
        density=density,
    )

    # Save as CSV
    harmonized.to_csv(out_path, index=False)
    print(f"-> {out_path}")

    return harmonized


Introduction
------------

This notebook size-harmonizes observations of atmospheric microplastic concentration and deposition from the sampled size range to the 0.1-100 µm size range used in GEOS-Chem simulations.

Extrapolation equations
-----------------------

Note that here we define $\alpha$ as the absolute value of the power law exponent (e.g. 2) and prefix it with a negative sign. This differs slightly from @Segur2026, who define $a$ as the value of the exponent (e.g. -2).

### Extrapolating observed number of particles to a new size range

We assume that atmospheric microplastics follow a power law size distribution. Specifically, the number of particles whose size is in the range \[$x$, $x + \text{d}x$\] as $\text{d}x \rightarrow 0$ is given by:

$$
n(x) = C x^{-\alpha}
$$

where $C$ is a scaling factor related to the total number of particles in the sample and the exponent $\alpha$ adjusts the relative frequency of small vs. large particles. Since particle size cannot be zero or infinity, we implicitly assume that the power law size distribution only holds over some size range $[x_{Min}, x_{Max}]$ which contains our size range of interest.

The number of particles in a size range $[x_0, x_1]$ can be computed by integration:

$$
\begin{split}
N_{[x_0, x_1]} &= \int_{x_0}^{x_1} C x^{-\alpha} \text{d}x \\
               &= \frac{C}{1 - \alpha} (x_1^{1 - \alpha} - x_0^{1 - \alpha})
\end{split}
$$

Given $\alpha$ and the observed number of particles in a known size range $[x_0, x_1]$, we can write the scaling factor $C$ as:

$$
C = \frac{N_{[x_0, x_1]} (1 - \alpha)}{x_1^{1 - \alpha} - x_0^{1 - \alpha}}
$$

Substituting this expression for $C$ in the previous equation, we can extrapolate the number of particles in a different size range $[x_{min}, x_{max}]$:

$$
N_{[x_{min}, x_{max}]} = N_{[x_0, x_1]}
                         \frac{x_{max}^{1 - \alpha} - x_{min}^{1 - \alpha}}
                              {x_1^{1 - \alpha} - x_0^{1 - \alpha}}
$$

### Estimating particle mass in a new size range

The mass of a single particle of size $x$ is:

$$
m(x) = \rho \times k \times x^3
$$

where $\rho$ is the particle density and $k$ is a shape factor that relates the volume of a single particle to the volume of a cube with edge $x$. For example, the volume of a sphere is $\frac{pi}{6} x^3$ so the shape factor $k$ for a spherical particle is $\frac{pi}{6} \approx 0.5236$.

Assuming that particle shape does not vary with size (i.e. the ratio between particle length, width, and height is constant), the total mass of all particles in a size range $[x_{min}, x_{max}]$ is:

$$
\begin{split}
M_{[x_{min}, x_{max}]} &= \int_{x_{min}}^{x_{max}} \rho \times k \times x^3
                          \times C \times x^{-\alpha} \text{d}x \\
                       &= \int_{x_{min}}^{x_{min}} \rho \times k \times C
                          \times x^{3 - \alpha} \text{d}x \\
                       & = \rho \times k \times \frac{C}{4 - \alpha}
                           (x_{max}^{4 - \alpha} - x_{min}^{4 - \alpha}) \\
                       & = C \frac{\rho k}{4 - \alpha}
                           (x_{max}^{4 - \alpha} - x_{min}^{4 - \alpha})
\end{split}
$$

Given $\rho$, $k$, $\alpha$ and the observed number of particles in a reference size range $[x_0, x_1]$, we compute the value of $C$ using the expression above and use this equation to extrapolate the mass in any size range.

Size-harmonize @Fu2023 observations
-----------------------------------

First, we use data from MPsizeBase [@Sonke2025; @Segur2026], a large database of size-resolved microplastic observations, to compute the median value of $\alpha$, the power law exponent, for atmospheric microplastics. Note that MPsizeBase records $\alpha$ as negative:

In [2]:
mpsb_path = os.path.join(PROCESSED_DIR, "mpsizebase_atmo.csv")
mpsb = pd.read_csv(mpsb_path)
(
    mpsb.loc[
        mpsb["sub_type1"].eq("Aerosol")
        & mpsb["sub_type2"].eq("Outdoor")
        & mpsb["shape"].eq("Fragments")
    ]
    .groupby(["sub_type1", "shape"])
    .agg(
        {
            "doi": "nunique",
            "psd": "nunique",
            "psd_id": "nunique",
            "alpha": ["median", "min", "max", "mean", "std"],
        }
    )
)

doi     psd  psd_id     alpha                      \
                    nunique nunique nunique    median       min       max   
sub_type1 shape                                                             
Aerosol   Fragments       8       8      16 -2.833286 -4.305377 -1.936731   

                                         
                         mean       std  
sub_type1 shape                          
Aerosol   Fragments -2.619756  0.623049

We then compute the edges of the size bins represented by the spherical simulation tracers if each tracer size (diameter) is the volume-weighted mean size of a bin:

$$
\begin{split}
           v(x) &= kx^3 \\
        \bar{v} &= V_{\text{total}} / N_{\text{total}} \\
k x_{\bar{v}}^3 &= V_{\text{total}} / N_{\text{total}} \\
k x_{\bar{v}}^3 &= \frac{
                     \frac{kC}{4 - \alpha} (x_1^{4 - \alpha} - x_0^{4 - \alpha})
                   }
                   {
                     \frac{C}{1 - \alpha} (x_1^{1 - \alpha} - x_0^{1 - \alpha})
                   } \\
  x_{\bar{v}}^3 &= \frac{1 - \alpha}{4 - \alpha} \cdot
                   \frac{
                     x_1^{4 - \alpha} - x_0^{4 - \alpha}
                   }
                   {
                     x_1^{1 - \alpha} - x_0^{1 - \alpha}
                   } \\
\end{split}
$$

We numerically solve the final expression for $x_1$, each bin's upper edge, given the lower edge $x_0$, $\alpha$ = 2.8, and the tracer diameters $x_{\bar{v}} \in \{0.3, 2.5, 7, 15, 35, 70\}$ µm. We set the first bin's lower edge to 0.092 µm, which gives a total simulation size range of [0.092 µm, 99.5 µm].

In [3]:
alpha = 2.8  # MPsizeBase records alpha with negative sign
xmin = 0.092

# Compute bin edges [µm] such that tracer sizes are volume-weighted bin mean diameters
bin_edges = powerlaw_compute_bin_edges(
    xmin=xmin, vol_mean_sizes=PLASTIC_SIZES, alpha=alpha, figs=3
)
bin_edges = bin_edges * ureg("µm")
bin_edges

Magnitude,[0.092 1.3 5.27 9.46 24.9 50.4 99.5]
Units,micrometer


We load corrected observations of atmsopheric microplastic concentration and deposition and size-harmonize the particle number and mass to the simulation size range.

When computing mass, we assume all microplastics are fragments that can be approximated by an ellipsoid with height = 0.68 length and width = 0.4 height (shape factor $\approx$ 0.097) [@Sonke2025; @Segur2026]. Following @Fu2023, we assume plastic density is 1 g/cm^3^.

In [4]:
shape_factor = PLASTIC_SHAPE_FACTORS["fragments"]
density = 1 * ureg("g/cm3")

### Concentration

In [5]:
# size-harmonize concentration
conc_harmonized = harmonize_or_load(
    obs_name="obs_revised_concentration.csv",
    out_name="obs_size-harmonized_concentration.csv",
    size_min=bin_edges[0],
    size_max=bin_edges[-1],
    alpha=alpha,
    shape_factor=shape_factor,
    density=density,
    overwrite=OVERWRITE,
)

-> results/obs_size-harmonized_concentration.csv


In [6]:
clean_styler(
    conc_harmonized.groupby(["study", "doi", "setting"])
    .agg(
        {
            "lat": "count",
            "size_low": "min",
            "size_high": "max",
            "particle/m3": "mean",
            "ug/m3": "mean",
            "ng/particle": "mean",
            "alpha": "mean",
            "shape_factor": "mean",
        }
    )
    .rename(columns={"lat": "n_obs"})
    .style.format(precision=4)
    .format(precision=3, subset=["size_low"])
    .format(precision=1, subset=["size_high"])
    .format(precision=0, subset=["particle/m3"])
    .format(precision=1, subset=["alpha"])
)

,,,n_obs,size_low,size_high,particle/m3,ug/m3,ng/particle,alpha,shape_factor
study,doi,setting,,,,,,,,
Abbasi2019,https://doi.org/10.1016/j.envpol.2018.10.039,land,1,0.092,99.5,3042,0.0015,0.0005,2.8,0.0968
Allen2020,https://doi.org/10.1371/journal.pone.0232746,land,1,0.092,99.5,3862,0.0019,0.0005,2.8,0.0968
Allen2021,https://doi.org/10.1038/s41467-021-27454-7,land,1,0.092,99.5,164,0.0001,0.0005,2.8,0.0968
Caracci2023,https://doi.org/10.1016/j.jhazmat.2023.131036,ocean,10,0.092,99.5,7,0.0000,0.0005,2.8,0.0968
Ding2021,https://doi.org/10.1016/j.atmosenv.2021.118389,ocean,1,0.092,99.5,2937,0.0015,0.0005,2.8,0.0968
Dris2017,https://doi.org/10.1016/j.envpol.2016.12.013,land,1,0.092,99.5,75574,0.0374,0.0005,2.8,0.0968
Gaston2020,https://doi.org/10.1177/0003702820920652,land,1,0.092,99.5,149286,0.0738,0.0005,2.8,0.0968
Liu2019,https://doi.org/10.1021/acs.est.9b03427,ocean,3,0.092,99.5,3022,0.0015,0.0005,2.8,0.0968
Liu2020,https://doi.org/10.1016/j.jhazmat.2020.123223,land,9,0.092,99.5,1280,0.0006,0.0005,2.8,0.0968


### Deposition

In [7]:
# Size-harmonize deposition
depo_harmonized = harmonize_or_load(
    obs_name="obs_revised_deposition.csv",
    out_name="obs_size-harmonized_deposition.csv",
    size_min=bin_edges[0],
    size_max=bin_edges[-1],
    alpha=alpha,
    shape_factor=shape_factor,
    density=density,
    overwrite=OVERWRITE,
)

-> results/obs_size-harmonized_deposition.csv


In [8]:
clean_styler(
    depo_harmonized.groupby(["study", "doi", "setting"])
    .agg(
        {
            "lat": "count",
            "size_low": "min",
            "size_high": "max",
            "particle/m2/d": "mean",
            "t/km2/yr": "mean",
            "ng/particle": "mean",
            "alpha": "mean",
            "shape_factor": "mean",
        }
    )
    .rename(columns={"lat": "n_obs"})
    .style.format(precision=4)
    .format(precision=3, subset=["size_low"])
    .format(precision=1, subset=["size_high"])
    .format(precision=0, subset=["particle/m2/d"])
    .format(precision=1, subset=["alpha"])
)

,,,n_obs,size_low,size_high,particle/m2/d,t/km2/yr,ng/particle,alpha,shape_factor
study,doi,setting,,,,,,,,
Allen2019,https://doi.org/10.1038/s41561-019-0335-5,land,1,0.092,99.5,1688451,0.0003,0.0005,2.8,0.0968
Brahney2020,https://doi.org/10.1126/science.aaz5819,land,11,0.092,99.5,117264,0.0000,0.0005,2.8,0.0968
Cai2017,https://doi.org/10.1007/s11356-017-0116-x,land,1,0.092,99.5,579906,0.0001,0.0005,2.8,0.0968
Dris2016,https://doi.org/10.1016/j.marpolbul.2016.01.006,land,1,0.092,99.5,9224923,0.0017,0.0005,2.8,0.0968
Klein2019,https://doi.org/10.1016/j.scitotenv.2019.05.405,land,1,0.092,99.5,4914706,0.0009,0.0005,2.8,0.0968
Liu2020,https://doi.org/10.1016/j.jhazmat.2020.123223,land,4,0.092,99.5,544380,0.0001,0.0005,2.8,0.0968
Roblin2020,https://doi.org/10.1021/acs.est.0c04000,land,1,0.092,99.5,1257711,0.0002,0.0005,2.8,0.0968
Wright2020,https://doi.org/10.1016/j.envint.2019.105411,land,1,0.092,99.5,18561383,0.0034,0.0005,2.8,0.0968
Yuan2023,https://doi.org/10.1016/j.scitotenv.2023.161839,land,1,0.092,99.5,2149647,0.0004,0.0005,2.8,0.0968
